In [1]:
from parsers.lr.lr1 import LR1Parser
from parsers.lr.lr_parsing_engine import LREngine

from parsers.grammars.appel_grammar_parser import appel_to_ruleset

In [2]:
grammar = [
    "P -> L",
    "S -> id := id",
    "S -> if id then S",
    "S -> if id then S else S",
    "L -> S",
    "L -> L ; S",
]

In [3]:
p = LR1Parser(appel_to_ruleset(grammar)).to_lalr()

In [4]:
print(p)

P─── L $end
S─┬─ id := id
  ├─ if id then S
  └─ if id then S else S
L─┬─ S
  └─ L ; S



In [ ]:
# TODO: Fix ASAP, sometimes the table generates in totally different way
#       Idk what can be the reason

print(p.to_tabulate())

┌────┬────────┬─────┬──────┬──────┬─────────┬────────┬──────┬─────┬─────┬─────┐
│    │ then   │ ;   │ :=   │ id   │ else    │ $end   │ if   │ P   │ S   │ L   │
├────┼────────┼─────┼──────┼──────┼─────────┼────────┼──────┼─────┼─────┼─────┤
│  0 │        │     │      │ s2   │         │        │ s1   │     │ g8  │     │
├────┼────────┼─────┼──────┼──────┼─────────┼────────┼──────┼─────┼─────┼─────┤
│  1 │        │     │      │ s15  │         │        │      │     │     │     │
├────┼────────┼─────┼──────┼──────┼─────────┼────────┼──────┼─────┼─────┼─────┤
│  2 │        │     │ s3   │      │         │        │      │     │     │     │
├────┼────────┼─────┼──────┼──────┼─────────┼────────┼──────┼─────┼─────┼─────┤
│  3 │        │     │      │ s17  │         │        │      │     │     │     │
├────┼────────┼─────┼──────┼──────┼─────────┼────────┼──────┼─────┼─────┼─────┤
│  5 │        │     │      │ s2   │         │        │ s1   │     │ g9  │ g6  │
├────┼────────┼─────┼──────┼──────┼─────

In [ ]:
cell = list(p.parsing_table[0]["else"])
cell[:1]
p.parsing_table[0]["else"] = set(cell[:1])

In [14]:
print(p.to_tabulate())

┌────┬──────┬────────┬────────┬──────┬─────┬──────┬────────┬─────┬─────┬─────┐
│    │ :=   │ then   │ else   │ if   │ ;   │ id   │ $end   │ P   │ L   │ S   │
├────┼──────┼────────┼────────┼──────┼─────┼──────┼────────┼─────┼─────┼─────┤
│  0 │      │        │        │ s11  │     │ s14  │        │     │ g10 │ g17 │
├────┼──────┼────────┼────────┼──────┼─────┼──────┼────────┼─────┼─────┼─────┤
│  1 │      │ s9     │        │      │     │      │        │     │     │     │
├────┼──────┼────────┼────────┼──────┼─────┼──────┼────────┼─────┼─────┼─────┤
│  2 │      │        │        │ s11  │     │ s14  │        │     │     │ g15 │
├────┼──────┼────────┼────────┼──────┼─────┼──────┼────────┼─────┼─────┼─────┤
│  3 │      │        │ r3     │      │ r3  │      │ r3     │     │     │     │
├────┼──────┼────────┼────────┼──────┼─────┼──────┼────────┼─────┼─────┼─────┤
│  5 │      │        │ s7     │      │ r2  │      │ r2     │     │     │     │
├────┼──────┼────────┼────────┼──────┼─────┼──────┼─

In [6]:
p.print_rules_and_states()

---=== Rules ===---
0000 P -> L $end
0001 S -> id := id
0002 S -> if id then S
0003 S -> if id then S else S
0004 L -> S
0005 L -> L ; S

---=== States ===---
state 0
    S -> if id then . S else S ($end ;)
    S -> . if id then S ($end ; else)
    S -> . id := id ($end ; else)
    S -> . if id then S else S ($end ; else)
    S -> if id then . S ($end ;)

state 1
    S -> if . id then S else S ($end ; else)
    S -> if . id then S ($end ; else)

state 2
    S -> id . := id ($end ; else)

state 3
    S -> id := . id ($end ; else)

state 5
    S -> . id := id ($end ;)
    S -> . if id then S else S ($end ;)
    L -> . S ($end ;)
    L -> . L ; S ($end ;)
    P -> . L $end
    S -> . if id then S ($end ;)

state 6
    P -> L . $end
    L -> L . ; S ($end ;)

state 7
    S -> if id then S else S . ($end ;)

state 8
    S -> if id then S . else S ($end ; else)
    S -> if id then S . ($end ; else)

state 9
    L -> S . ($end ;)

state 10
    S -> . if id then S ($end ; else)
    S -> . id :

In [7]:
e = LREngine(p)

In [8]:
code_sample = """
if id then     
    if id then
        id := id
    else
        id := id ;
        id := id $
""".split()
code_sample

['if',
 'id',
 'then',
 'if',
 'id',
 'then',
 'id',
 ':=',
 'id',
 'else',
 'id',
 ':=',
 'id',
 ';',
 'id',
 ':=',
 'id',
 '$']

In [9]:
def print_iter(stack, states, symbol, action):
    for el in stack:
        print_el = next(iter(el.keys())) if isinstance(el, dict) else el
        print(print_el, end=", ")
    print(f" {symbol=}, {states[-1]=}, {action=}")

In [10]:
import json

In [11]:
out = e.parse(code_sample, iteration_callback=print_iter)
print(json.dumps(out, indent=2))

 symbol='if', states[-1]=5, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=1)
if,  symbol='id', states[-1]=1, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=15)
if, id,  symbol='then', states[-1]=15, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=0)
if, id, then,  symbol='if', states[-1]=0, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=1)
if, id, then, if,  symbol='id', states[-1]=1, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=15)
if, id, then, if, id,  symbol='then', states[-1]=15, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=0)
if, id, then, if, id, then,  symbol='id', states[-1]=0, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=2)
if, id, then, if, id, then, id,  symbol=':=', states[-1]=2, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=3)
if, id, then, if, id, then, id, :=,  symbol='id', states[-1]=3, action=LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=17)
if, id, then, if, id, then, id, :=, id,  symbol='else', states[-1]=17, action=LRAc

AssertionError: actions={LRAction(type_=<LRActionEnum.SHIFT: 's'>, to=10), LRAction(type_=<LRActionEnum.REDUCE: 'r'>, to=2)}, self.curr_state()=8, sym='else'

In [ ]:
p.get_starting_state_idx()